# CytoVI Sub-Annotation — CD4/CD8 disambiguation

Take cells labelled `CD4/CD8` (double-positive / ambiguous) in `adata_cytovi_annotated_compat.h5ad`,
re-cluster on the existing CytoVI latent, and reassign each sub-cluster to either `CD4` or `CD8`
based on denoised marker expression. Writes the updated annotation back into `cell_type_annot`.

In [ ]:
%load_ext autoreload
%autoreload 2

import shutil
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from pathlib import Path

sc.set_figure_params(dpi=100, frameon=False)
plt.rcParams['figure.max_open_warning'] = 0

CACHE_DIR        = Path('/home/projects/nyosef/zvise/PixelGen/PixelGen/New_Data/cache')
ANNOTATED_CACHE  = CACHE_DIR / 'adata_cytovi_annotated_compat.h5ad'
BACKUP_CACHE     = CACHE_DIR / 'adata_cytovi_annotated_compat.pre_subannot.h5ad'

SUB_LEIDEN_RES   = 0.5

## 1. Load annotated data

In [ ]:
adata = sc.read_h5ad(ANNOTATED_CACHE)
print(adata)
print('\ncell_type_annot counts:')
print(adata.obs['cell_type_annot'].value_counts())

## 2. Subset to CD4/CD8 ambiguous cells

In [ ]:
AMBIG_LABEL = 'CD4/CD8'
mask_ambig = (adata.obs['cell_type_annot'] == AMBIG_LABEL).values
adata_sub = adata[mask_ambig].copy()
print(f'Ambiguous CD4/CD8 cells: {adata_sub.n_obs}')
print(adata_sub.obs['cell_system'].value_counts())

## 3. Re-cluster on CytoVI latent

Use the existing `X_CytoVI` representation — no need to retrain on this small subset.

In [ ]:
sc.pp.neighbors(adata_sub, use_rep='X_CytoVI', n_neighbors=15)
sc.tl.umap(adata_sub, min_dist=0.3)
sc.tl.leiden(adata_sub, resolution=SUB_LEIDEN_RES, key_added='sub_leiden')
print(f'Sub-clusters: {adata_sub.obs["sub_leiden"].nunique()}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sc.pl.umap(adata_sub, color='sub_leiden',  ax=axes[0], show=False, title='Sub-leiden clusters')
sc.pl.umap(adata_sub, color='cell_system', ax=axes[1], show=False, title='Cell system')
plt.tight_layout()
plt.show()

In [ ]:
key_markers = [m for m in ['CD3e', 'CD4', 'CD8', 'CD19', 'CD20'] if m in adata_sub.var_names]
sc.pl.umap(adata_sub, color=key_markers, layer='imputed', ncols=5, cmap='viridis')

## 4. Marker dotplot per sub-cluster

In [ ]:
marker_groups = {
    'T cell': ['CD3e', 'TCRab', 'CD5', 'CD7', 'CD2'],
    'CD4 T':  ['CD4'],
    'CD8 T':  ['CD8', 'CD57', 'KLRG1'],
    'B cell': ['CD19', 'CD20', 'CD22', 'IgM', 'IgD'],
}
markers_f = {k: [m for m in v if m in adata_sub.var_names] for k, v in marker_groups.items()}
markers_f = {k: v for k, v in markers_f.items() if v}

sc.pl.dotplot(
    adata_sub, var_names=markers_f, groupby='sub_leiden',
    layer='imputed', standard_scale='var',
    title='CytoVI — denoised marker expression per sub-cluster',
    figsize=(14, 5),
)

## 5. Annotate sub-clusters as CD4 / CD8

Inspect the dotplot above and fill in the dictionary. Allowed labels: `CD4`, `CD8`, `Doublets` (if a sub-cluster is clearly non-T or still ambiguous).

In [ ]:
# Fill in: 'CD4' | 'CD8' | 'CD4/CD8' | 'Doublets'
sub_annotation = {
    '0': 'CD8',
    '1': 'CD8',
    '2': 'CD4/CD8',
    '3': 'CD4',
    '4': 'CD4',
    '5': 'CD8',
   
}

In [ ]:
missing = set(adata_sub.obs['sub_leiden'].astype(str).unique()) - set(sub_annotation.keys())
assert not missing, f'Fill in labels for sub-clusters: {sorted(missing)}'
VALID_LABELS = {'CD4', 'CD8', 'CD4/CD8', 'Doublets'}
bad = {k: v for k, v in sub_annotation.items() if v not in VALID_LABELS}
assert not bad, f'Invalid labels: {bad}. Must be one of: {VALID_LABELS}'

adata_sub.obs['sub_cell_type'] = adata_sub.obs['sub_leiden'].astype(str).map(sub_annotation)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sc.pl.umap(adata_sub, color='sub_cell_type', ax=axes[0], show=False, title='Sub-annotation')
sc.pl.umap(adata_sub, color='cell_system',   ax=axes[1], show=False, title='Cell system')
plt.tight_layout()
plt.show()

print(adata_sub.obs['sub_cell_type'].value_counts())

## 6. Write annotations back into the full AnnData and save

In [ ]:
# Make a working copy of the full adata and update the cell_type_annot column
adata.obs['cell_type_annot'] = adata.obs['cell_type_annot'].astype(str)
adata.obs.loc[adata_sub.obs_names, 'cell_type_annot'] = adata_sub.obs['sub_cell_type'].values
adata.obs['cell_type_annot'] = adata.obs['cell_type_annot'].astype('category')

print('Updated cell_type_annot counts:')
print(adata.obs['cell_type_annot'].value_counts())

sc.pl.umap(adata, color='cell_type_annot', title='Updated cell type annotation')

In [ ]:
# Back up the original (only on first run, never overwrites the backup)
if not BACKUP_CACHE.exists():
    shutil.copy2(ANNOTATED_CACHE, BACKUP_CACHE)
    print(f'Backup created -> {BACKUP_CACHE}')
else:
    print(f'Backup already exists -> {BACKUP_CACHE}')

adata.write_h5ad(ANNOTATED_CACHE)
print(f'Saved -> {ANNOTATED_CACHE}')